In [ ]:

# Install Required Packages
!pip install requests beautifulsoup4 pandas


In [ ]:

# Step 1: Import Libraries
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re

# Step 2: Define Constants
ADCC_EVENTS_URL = "https://adcc.smoothcomp.com/en/federation/176/events/past"
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"}
DEBUG_MODE = True  # Set to True for debugging, False for full run
DEBUG_LIMIT = 5   # Limit number of events for debugging


In [ ]:

# Step 3: Scrape the list of past ADCC Open events
def get_event_list():
    """Scrapes the list of past ADCC Open events from Smoothcomp."""
    response = requests.get(ADCC_EVENTS_URL, headers=HEADERS)
    soup = BeautifulSoup(response.text, 'html.parser')
    
    events = []
    event_cards = soup.find_all("div", class_="card-body")
    
    for card in event_cards:
        link_tag = card.find("a", href=True)
        if link_tag:
            event_url = "https://adcc.smoothcomp.com" + link_tag["href"]
            event_name = link_tag.text.strip()
            
            date = card.find("span", class_="date").text.strip() if card.find("span", class_="date") else "Unknown"
            location = card.find("span", class_="location").text.strip() if card.find("span", class_="location") else "Unknown"
            
            events.append({
                "Event Name": event_name,
                "Event URL": event_url,
                "Date": date,
                "Location": location
            })
            
        if DEBUG_MODE and len(events) >= DEBUG_LIMIT:
            break
    
    df = pd.DataFrame(events)
    df.to_csv("adcc_events.csv", index=False)
    print("Event list saved to adcc_events.csv")
    return df


In [ ]:

# Step 4: Scrape match data for a given event
def get_match_list(event_url):
    """Scrapes match data from a given event match list page."""
    response = requests.get(event_url + "/schedule/matchlist", headers=HEADERS)
    soup = BeautifulSoup(response.text, 'html.parser')
    
    matches = []
    match_rows = soup.find_all("tr", class_="match-row")
    
    for row in match_rows:
        cols = row.find_all("td")
        if len(cols) >= 5:
            match_id = row["data-id"] if row.has_attr("data-id") else "Unknown"
            competitor1 = cols[1].text.strip()
            competitor2 = cols[2].text.strip()
            result = cols[3].text.strip()
            details_url = "https://adcc.smoothcomp.com" + cols[0].find("a")["href"] if cols[0].find("a") else "Unknown"
            
            matches.append({
                "Match ID": match_id,
                "Competitor 1": competitor1,
                "Competitor 2": competitor2,
                "Result": result,
                "Match URL": details_url
            })
    
    return matches


In [ ]:

# Step 5: Scrape match data for all past ADCC Open events
def scrape_adcc_matches():
    """Scrapes match data for all past ADCC Open events and saves it to a CSV file."""
    event_df = get_event_list()
    all_matches = []
    
    for _, row in event_df.iterrows():
        event_name = row["Event Name"]
        event_url = row["Event URL"]
        print(f"Scraping matches for {event_name}: {event_url}")
        matches = get_match_list(event_url)
        all_matches.extend(matches)
        
        if DEBUG_MODE and len(all_matches) >= DEBUG_LIMIT:
            break
        
        time.sleep(1)  # Be polite to the server
    
    df = pd.DataFrame(all_matches)
    df.to_csv("adcc_matches.csv", index=False)
    print("Match data saved to adcc_matches.csv")


In [ ]:

# Step 6: Run the scraper
if __name__ == "__main__":
    scrape_adcc_matches()
